## MoE

- shared experts
- fine-grained experts
- router loss (2 variations)


nn.Sequential - automatically the input flows through the specified layers  
nn.ModuleList - you need to iterate through thel layers with a for loop


In [3]:
import torch.nn as nn
import torch
import torch.nn.functional as F
import einops

class Expert(nn.Module):

	def __init__(self, d_ff:int, d_embed:int):
		super().__init__()

		self.network = nn.Sequential( 
			nn.Linear(d_embed, d_ff), 
			nn.ReLU(),
			nn.Dropout(0.1), # hidden layer dropout, always after non-linearity, since ReLU 
			nn.Linear(d_ff, d_embed),
			nn.Dropout(0.1), # right before residual
		)

	def forward(self, x_in): 

		x_out = self.network(x_in) # residual stream gets added outside the block

		return x_out

class Router(nn.Module): 

	def __init__(self, num_experts:int, d_embed:int): 
		super().__init__()
		self.router = nn.Linear(d_embed, num_experts)

	def forward(self, x_in): 

		x_out = self.router(x_in)
		return x_out

MoE Block


A nice way to visualize what an MoE block is doing

<img src="https://awsdocs-neuron.readthedocs-hosted.com/en/latest/_images/moe-architecture-overview.png" width="600">

- x_in = [N tokens, d_embed] <- token input
- [token indexes,] <- which tokens have been assigned to this expert i
  - we get this by doing passing the input through a router, to give us -> [N tokens, num_experts]
  - then we get the topk experts -> [N tokens, topk]
  - from that, for token 1 we get [expert 1, expert 2] for example
  - then we find all the indices in this tensor where our expert i, is one of those topk - this would mean that token should pass thru this expert
  - [N tokens, relevant topk experts] == expert i, along dim=1, which gives a boolean matrix back that is the same size
  - Identify which tokens there is a True boolean for
  - non.zero() gives us the exact indices of each token where a True exists!
  - this returns us tensor[token_idx], tensor[ranking in the topk experts chosen for this token] -> dim=1 can also be interpreted as the expert index for that token

_Now its easy_

- use the [token indexes, ] tensor (AKA: token_idx) to pluck out all the tokens assigned to expert i from out input x_in
- GETTING THE EXPERT OUTPUTS:
  - now pass that batched input through expert(x_flat)
  - this is the batching of the tokens for expert i (see: BLOCK)

<img src="https://developer-blogs.nvidia.com/wp-content/uploads/2025/10/image6-png.webp" width="600">

- GETTING THE EXPERT WEIGHTS:
  - use the same token_idx tensor and the tensor[expert i's ranking for each selected token], to pluck out the softmaxxed logits given by the router (i.e. probability assigned to this expert for this token)
- WHAT WE HAVE NOW: [Relevant tokens to expert i, probability weight for each of these tokens], [Relevant tokens to expert i, expert i's output for each of these tokens]

- FINISHING MOVE
  - now multiply the expert outputs by the probability weights, and then store that output. That output is the contribution of expert i, to all N token's output values

_Now repeat_

- do this but loop over all experts since each will contribute to every token's output value


In [ ]:
class MoeBlock(nn.Module):

	def __init__(self, num_experts:int, d_embed:int, d_ff:int, topk:int, num_shared_experts:int=None): # all defaults must be at the end of init
		super().__init__()

		self.num_experts = num_experts

		# initialize the router 
		self.router = Router(num_experts=num_experts, d_embed=d_embed)

		# initialize the experts
		self.experts = nn.ModuleList([Expert(d_ff=d_ff, d_embed=d_embed) for _ in range(num_experts)])

		# top_k routing
		self.topk = topk

		# shared experts - `or 0` so the None default means "no shared experts" instead of crashing on range(None)
		self.shared_experts = nn.ModuleList([Expert(d_ff=d_ff, d_embed=d_embed) for _ in range(num_shared_experts or 0)])
		
	def forward(self, x_in):

		B, T, D = x_in.shape

		# use the router to find which experts should be routed to, then softmax 
		router_logits = self.router(x_in)

		full_router_probs = F.softmax(router_logits, dim=-1) # used to compute loss

		logit_values, score_indices = torch.topk(router_logits, k=self.topk) # returns a tuple

		# select the topk
		score_values = F.softmax(logit_values, dim=-1) # shaped (B, T, topk)
		
		# using the topk array, using the indices, index into the experts Module list, and pass x_in through that. Then sum all together, weighting the output by the value of the topk output
		# for this type of operation use x.index_add_(dim, index, source, *, alpha=1)
		
		# initialize the array to store each token
		x_flat = x_in.view(B*T, D) # (N d)
		idx_flat = score_indices.view(B*T, self.topk) # (N, topk) <- indices
		gates_flat = score_values.view(B*T, self.topk) # (N, topk) <- values
		
		out = torch.zeros((B * T, D)).to(x_in.device)

		# ideally, you would like to index into the expert array with the last_dimension of score_indices, creating a (B, T, top_k) matrix. However this would require replicating
		# the expert tensors which would be bad for memory, so instead we iteratively cycle through the experts, which hold the tensor that is expensive to hold in memory
		# it also makes sense to accumulate all the tokens assigned to an expert and do a batch matmul

		fraction = torch.zeros(len(self.experts)) # (N_experts,)
		probs = full_router_probs.mean(dim=0) # mean probs per expert across all tokens (N_experts,)

		for e, expert in enumerate(self.experts):

			# check if expert is one of the top_k
			token_idx, slot_idx = (idx_flat == e).nonzero(as_tuple=True) 
			# .nonzero output -> the columns store the index for dim0, dim1, so to index correctly you must zip the tensors
			# token_idx = dim0 token index, slot_idx = dim1 expert index, tensors are size (len(token_idx),)

			if token_idx.numel() == 0: # this expert does not attend to any tokens
				continue

			# now index into x_flat to pull out the tokens
			tokens = x_flat[token_idx] # (L, d_embed), where L = len(token_idx)

			# ok so now we can feed the tokens into the experts
			# but that output needs to be weighted by the score_values
			weights = gates_flat[token_idx, slot_idx] # (L,) <- this should hold the respective weights for each token assigned to this expert

			out.index_add_(dim=0, index=token_idx, source=weights.unsqueeze(-1) * expert(tokens)) # index in the seq_len dimension, weights get broadcasted and multiply along the dimensions

			fraction[e] = len(slot_idx) / (B*T*self.topk) # how many tokens were assigned to expert e // total number of possible slots

		# an empty ModuleList is falsy, so this loop simply does not run when there are no shared experts
		for expert in self.shared_experts: 
			
			# add shared experts! 
			out += expert(x_flat)

		moe_out = out.view(B, T, D)

		return moe_out, len(self.experts) * torch.sum(fraction * probs)

# Find the variance along the expert dimension -> (B, T, num_experts), sum across B, T and average

x_in = torch.rand(32, 1024, 256)
moeblock = MoeBlock(num_experts=12, d_ff=128, d_embed=256, topk=2, num_shared_experts=2)

x_out = moeblock(x_in)

MoE GPT


In [ ]:
# copy of MLA for the GPT code
# TODO: implement deepseek-v3 sparse attention
class MLA(nn.Module): 

	def __init__(self, q_latent_dim:int=12, kv_latent_dim:int=4, d_embed:int=256, num_heads:int=8, max_seq_len:int=1024): 
		super().__init__()

		self.q_latent = nn.Linear(d_embed, q_latent_dim)
		self.query = nn.Linear(q_latent_dim, d_embed)
		self.kv_latent = nn.Linear(d_embed, kv_latent_dim)
		self.key = nn.Linear(kv_latent_dim, d_embed)
		self.value = nn.Linear(kv_latent_dim, d_embed)
		self.num_heads = num_heads

		self.w_out = nn.Linear(d_embed, d_embed)
		
		self.register_buffer('causal_mask', torch.tril(torch.ones(max_seq_len, max_seq_len)).bool(), persistent=False) # this is a nn.Module method that registers a self.causal_mask but on CUDA

	def forward(self, x_in:int, block_kv_cache=None): 
		is_prefill = block_kv_cache is None

		b, t, d = x_in.shape

		q_latent = self.q_latent(x_in)
		query_latent = self.query(q_latent)
		query = rearrange(query_latent, 'b q (n d) -> b n q d', n=self.num_heads)

		kv_latent = self.kv_latent(x_in)

		if not is_prefill:
			kv_latent = torch.cat((block_kv_cache['kv_latent'], kv_latent), dim=1) # (B, T, low_rank) -> concat along the seq_len dimension
			# then (B, T, low_rank) * (low_rank, d_embed) -> (B, T, d_embed) which reassembles the full key, value matrices
		
		# reassemble full kv matrices (costs O(2*latent_dim*embed_dim))
		key_latent = self.key(kv_latent)
		value_latent = self.value(kv_latent)
		
		# reshape into heads
		key = rearrange(key_latent, 'b k (n d) -> b n k d', n=self.num_heads)
		value = rearrange(value_latent, 'b v (n d) -> b n v d', n=self.num_heads)

		# assemble causal self-attention matrix
		logits = torch.einsum('b n q d, b n k d -> b n q k', query, key) / ((d//self.num_heads)**0.5) # divide by the head_dim of the

		if is_prefill:
			logits = logits.masked_fill(~self.causal_mask[:t, :t], float('-inf')) # where the masked_fill isn't true, replace it with -inf

		scores = torch.softmax(logits, dim=-1)
		attention = torch.einsum('b n q k, b n k d -> b n q d', scores, value)
		attention = rearrange(attention, 'b n q d -> b q (n d)')
		
		attention_output = self.w_out(attention)

		block_kv_cache = {
			"kv_latent": kv_latent,
		}
		
		return attention_output, block_kv_cache

<img src="https://substackcdn.com/image/fetch/$s_!W4Qo!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2Ff4b97110-d705-4531-b4af-4f87187a8dea_1393x1394.png" width="400">


In [ ]:
from transformers import AutoTokenizer

class MoeTransformer(nn.Module): 

	def __init__(self, d_embed:int, d_ff:int, num_experts:int, topk:int, num_shared_experts:int=None, block_kv_cache=None):
		super().__init__()
		
		self.moe_block = MoeBlock(num_experts=num_experts, d_embed=d_embed, d_ff=d_ff, topk=topk, num_shared_experts=num_shared_experts)
		self.mla = MLA(d_embed=d_embed)

		# we'll use pre-norm as per DSv3
		self.ln1 = nn.LayerNorm()
		self.ln2 = nn.LayerNorm()

	def forward(self, x_in):

		norm_x_in = ln1(x_in) # first layernorm to pass into att
		attention, kv_cache = self.mla(norm_x_in, block_kv_cache=block_kv_cache) # att

		x_r1 = x_in + attention # residual connection 1
		
		norm_x_r1 = ln2(x_r1) # normalize before moe
		out_moe, lb_loss = self.MoeBlock(norm_x_r1) # moe output

		x_r2 = out_moe + x_r1 # residual connection 2

		return x_r2, kv_cache, lb_loss


class MoeGPT(nn.Module):

	def __init__(self, d_embed:int, d_ff:int, num_experts:int, topk:int, num_transformer_blocks: int, num_shared_experts:int=None):
		super().__init__()

		self.tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-V3", trust_remote_code=True)
		self.embedding = nn.Embedding(vocab_size=len(tokenizer), embedding_dim=256)

		self.layers = nn.ModuleList([MoeTransformer(d_embed=d_embed, d_ff=d_ff, num_experts=num_experts, topk=topk, num_shared_experts=num_shared_experts) for _ in range(num_transformer_blocks)])

	def forward(self, x_in):

		

In [ ]:
def compute_moe_loss(response_token_ids: list[int], target_token_ids: list[int], w_lb:float, w_z:float):

	loss = 0
	# normal SFT loss

	# MoE load balancing loss

	# MoE logits loss

	pass

def compute_dsv3_moe_loss(response_token_ids: list[int], target_token_ids: list[int], w_lb:float, w_z:float):

	# normal SFT loss

	# MoE auxiliary loss
	pass

epochs = 10

def train():

	for _ in range(epochs)
	pass

Latent MoE (Kimi K3)
